# Tutorial: training and predicting on 1000 Genomes

Here we train Diet Networks to predict population from genotypes, and use them to predict, on a small dataset that is included in the repository.
It contains 480 individuals of the 1000 Genomes Project (1KGP), 20 from each of 24 populations, at 2,000 SNPs.
Everything runs in under a minute on a laptop, without a GPU or a network connection.

The models are much less accurate than the published ones, which were trained on 229,986 SNPs and more individuals.
The aim is to show what each step does and what it produces.

To run the notebook yourself, clone the repository, run `uv sync --frozen --extra docs`, and open `docs/tutorial.ipynb`.
The first cell defines `dietnet()`, which runs a command as you would in a terminal and prints its output.

In [ ]:
import os
import subprocess
import sys
import tempfile
from pathlib import Path

import h5py
import numpy as np
import pandas as pd

# The fixtures are in the repository, next to docs/.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tests" / "fixtures").is_dir())
FIXTURES = REPO / "tests" / "fixtures"
PANEL = FIXTURES / "kgp_mini"
LABELS = FIXTURES / "kgp_mini_labels.tsv"

# Everything this notebook writes goes to a scratch directory.
os.chdir(tempfile.mkdtemp(prefix="dietnet-tutorial-"))


def dietnet(*args, tail=None):
    """Run a dietnet command; print its output (only the last `tail` lines if given)."""
    result = subprocess.run(
        [sys.executable, "-m", "dietnet.cli", *map(str, args)],
        capture_output=True,
        text=True,
    )
    here = os.path.realpath(os.getcwd())
    output = (result.stdout + result.stderr).replace(here + os.sep, "")
    lines = [line for line in output.splitlines() if line.strip(" -")]
    print("\n".join(lines[-tail:] if tail else lines))
    if result.returncode != 0:
        raise RuntimeError(f"dietnet {args[0]} exited with status {result.returncode}")

In [ ]:
dietnet("info")

## The inputs

We need the genotypes, as PLINK files, and the population of each individual.
The labels are in a table with a header row, then one row per individual:

In [ ]:
labels = pd.read_csv(LABELS, sep="\t")
print(labels.head(3), "\n")
print(labels.iloc[:, 1].value_counts().describe()[["count", "min", "max"]])

In [ ]:
pd.read_csv(f"{PANEL}.bim", sep="\t", header=None,
            names=["chrom", "snp", "cm", "pos", "a1", "a2"]).head(3)

## 1. Create the dataset

`create-dataset` combines the genotypes and the labels into the files that the next steps read.
The genotype of an individual at a SNP is the number of copies of one of its alleles (A1 in the `.bim`), and the name of each SNP ends with this allele.
The genotypes are stored in a packed file next to the dataset.

In [ ]:
dietnet("create-dataset", "--plink", PANEL, "--labels", LABELS, "--out", "dataset.h5")

with h5py.File("dataset.h5") as f:
    for key in ("inputs", "samples", "snp_names", "class_labels", "class_label_names"):
        print(f"{key:18s} {f[key].dtype}  {f[key].shape}")
    print("first SNP names:", np.array(f["snp_names"][:2]).astype(str))

## 2. Split into folds

`partition` splits the individuals into five folds.
In each fold, 60% of the individuals of every population are used to train a model, 20% to choose when to stop training, and 20% to test it.

In [ ]:
dietnet("partition", "--dataset", "dataset.h5", "--out", "partition.npz", "--seed", 3, tail=1)

folds = np.load("partition.npz", allow_pickle=True)["folds_indexes"]
pd.DataFrame([[len(s) for s in fold] for fold in folds],
             columns=["train", "valid", "test"]).rename_axis("fold")

## 3. Compute the SNP features and statistics

The next two steps use only the training individuals of each fold.

`generate-embedding` computes the SNP features: for each SNP, the frequencies of genotypes 0, 1 and 2 in each population.
The auxiliary network uses them to compute the weights of the SNP (see [Training](../training/#free-snp-weights)).

`compute-input-stats` computes the mean and standard deviation of each SNP, which are used to standardize the genotypes.

In [ ]:
dietnet("generate-embedding", "--dataset", "dataset.h5", "--partition", "partition.npz",
        "--out", "embedding.npz", tail=1)
dietnet("compute-input-stats", "--dataset", "dataset.h5", "--partition", "partition.npz",
        "--out", "stats.npz", tail=1)

print("emb:", np.load("embedding.npz")["emb"].shape, "(folds x SNPs x 3 genotypes x 24 populations)")
stats = np.load("stats.npz")
print({key: stats[key].shape for key in stats.files})

## 4. Train

`train` fits one model for each seed and fold.
Here we train two models, with one seed on two folds.
The config describes a network that is small enough to train in seconds; the published one is in [Training](../training/#inputs).

In [ ]:
Path("config.yaml").write_text("""\
batch_size: 32
epochs: 60
input_dropout: 0.5
dropout_main: 0.5
lr_aux: 0.001
lr_main: 0.001
learning_rate_annealing: 0.99
nb_hidden_u_aux: [64, 64]
nb_hidden_u_main: [64]
patience: 60
seed: 1
uniform_init_limit: 0.02
""")

dietnet("train", "--dataset", "dataset.h5", "--partition", "partition.npz",
        "--embedding", "embedding.npz", "--input-stats", "stats.npz",
        "--config", "config.yaml", "--exp-dir", "exp", "--seeds", 1, "--folds", "0,1")
sorted(str(p) for p in Path("exp/packages").rglob("*") if p.is_file())

Each model is saved as a package, which contains what is needed to predict with it and a record of how it was trained:

In [ ]:
from dietnet import package

packages = package.find("exp/packages")
{k: v for k, v in packages[0].metadata.items() if k not in ("classes", "results_dir")}

`check` reports the accuracy of each model on the individuals it was not trained on.
Choosing one of the 24 populations at random would give 1/24, or about 0.04.

In [ ]:
dietnet("check", "--packages", "exp/packages")

## 5. Predict

`predict` applies the models to a cohort, and combines their predictions.
Here the cohort is the dataset itself, so every SNP is found.
For another cohort, the line of counts shows how many SNPs were found, and how (see the [Quickstart](../quickstart/#2-predict)).

In the result, `probability` is the average over the two models, and `model_agreement` is the fraction of them that chose the same population:

In [ ]:
dietnet("predict", "--model", "exp/packages", "--plink", PANEL, "--out", "predictions.tsv",
        "--device", "cpu")
predictions = pd.read_csv("predictions.tsv", sep="\t")
predictions.head()

In [ ]:
dietnet("check", "--predictions", "predictions.tsv", "--labels", LABELS, tail=1)

This accuracy is much higher than the one reported by `check` above, but it is misleading: most of these individuals were used to train the models.
The accuracy on held-out individuals is the one to report.

The same predictions can be made in Python, which also gives the scores of each model:

In [ ]:
from dietnet import predict

matched = predict.matched_dataset(packages, PANEL, "work")
result = predict.predict(packages, matched, device="cpu")
print("scores:", result.scores.shape, "(models x samples x classes)")
print("scale:", result.scale)
result.alignment

## 6. Change the SNP features

`build-embedding` builds the SNP features from a list of blocks (see [SNP features](../features/)).
Here we add 16 random features to the published ones.
A model trained with them is a useful control: if another block of features improves on it, the improvement does not come only from having more features.

In [ ]:
Path("features.yaml").write_text("""\
blocks:
  - provider: class_genotype_freq
  - provider: random
    columns: 16
""")
dietnet("build-embedding", "--dataset", "dataset.h5", "--partition", "partition.npz",
        "--spec", "features.yaml", "--out", "embedding_plus_random.npz")

## 7. Train the baseline

The MLP learns the weights of its first layer directly, without the auxiliary network and so without SNP features.
It is the comparison point for any change to the features:

In [ ]:
dietnet("train", "--model", "mlp", "--dataset", "dataset.h5", "--partition", "partition.npz",
        "--input-stats", "stats.npz", "--config", "config.yaml", "--exp-dir", "exp_mlp",
        "--seeds", 1, "--folds", "0,1")
dietnet("check", "--packages", "exp_mlp/packages")